## Deep learning - trading signals using recurrent neural networks


In this notebook we will apply recurrent neural networks for portfolio optimization by forecasting returns of US stocks based on daily data. We will use a neural network for predicting returns and compare it with linear predictors. Then we use the extracted signals in a time-series momentum strategy.

We start by importing relevant packages.

In [ ]:
import sys
sys.path.append('..')  # the notebook lives in notebooks/, this lets us import from src/

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# my own helpers, factored out so the notebook stays readable
from src.data import load_prices, load_factors
from src.features import make_windows
from src.backtest import long_short_backtest, sharpe_ratio, sharpe_standard_error

# same seeds as the lecture
np.random.seed(1)
torch.manual_seed(1);

Load necessary datasets.

In [ ]:
# daily adjusted close prices, one column per stock
# load_prices reads the course file sp500stocks.csv from data/ (see data/README.md)
prices = load_prices(start="2011-01-01", end="2023-12-31")
print(prices.shape)  # (days, stocks)

In [ ]:
# Fama French 3 factors + momentum, straight from Ken French's files
# these are in PERCENT simple returns, we convert them further down
factors = load_factors()
factors.tail()

## Data preparation

First we load daily price data for the past 13 years.

#### Load the daily price data

In [ ]:
# first look: which dates and how many stocks
# load_prices drops any stock with a missing price, so every column is complete
print("first day:", prices.index[0].date())
print("last day: ", prices.index[-1].date())
print("number of stocks:", prices.shape[1])
prices.iloc[:5, :6]

In [ ]:
# log returns, annualised by multiplying by 252 trading days
# log returns add up over time, which makes averaging over windows clean later on
returns = np.log(prices).diff().dropna() * 252
returns.iloc[:5, :6]

In [ ]:
# sanity check on the scale: annualised mean and volatility per stock
# the daily numbers are already x252, so the usual annual vol is std / sqrt(252)
ann_mean = returns.mean()
ann_vol = returns.std() / np.sqrt(252)

print(f"median annual mean log return: {ann_mean.median():.3f}")
print(f"median annual volatility:      {ann_vol.median():.3f}")

In [ ]:
# plot the first 15 stocks, each divided by its first price so they all start at 1
(prices.iloc[:, :15] / prices.iloc[0, :15]).plot(figsize=(12, 5), legend=False,
                                               title="15 of the S&P 500 stocks (normalised)")
plt.ylabel("price / first price")
plt.show()

In [ ]:
# check for extreme daily moves (bad data, splits that were not adjusted, etc.)
# a single wrong price would show up as a huge return here
daily_simple = prices.pct_change().dropna()
print("largest daily move:", f"{daily_simple.abs().max().max():.1%}")
print("stocks with a daily move above 50%:", (daily_simple.abs() > 0.5).any().sum())

# NOTE: these are 444 stocks with a price on every day from 2011 to 2023, i.e. firms that
# survived the whole period. Firms that went bust or were taken over are missing
# (survivorship bias), which is worth remembering when we look at Sharpe ratios later.

## Baseline: Factor Model Linear Regression

As a benchmark we will use a linear model. We use an extension of the CAPM. Here, we do not only consider the market portfolio (as in the practical session on Tuesday), but also additional factors. That is, we will try to predict (excess) returns based on three factors: (excess) returns of a market portfolio, a "small-big" portfolio and "high-minus-low"  portfolio (long stocks with high book-to-market ratios, short stocks with
low ones). This is refered to as the Fama French three factor model.

As further comparison, we will also use a forth factor called momentum (Long in stocks that have recently performed well, short on stocks that have performed poorly).

Daily values for these factors can be found in the files "Fama-French 3 factor" [Daily] and "Momentum Factor (Mom)" [Daily]
on Ken French’s website. http://mba.tuck.dartmouth.edu/pages/faculty/ken.french/data_library.html#Research
We downloaded these here and load them from the file.

Note that the data from Ken French’s website is percentages of simple returns,
so we convert them to log returns and we annualize all the data.

In [ ]:
# Ken French gives percent simple returns -> convert to log returns, then annualise like the stocks
factors_log = np.log(1 + factors / 100) * 252

# keep only the days where we have stock returns (same as .reindex in the lecture)
factors_log = factors_log.reindex(returns.index)
print("missing factor values:", factors_log.isna().sum().sum())

# careful: one of the stocks has the ticker "RF" (Regions Financial), the same name as the
# risk free rate column. Keeping the factors in their own DataFrame avoids mixing them up.
print("is there a stock called RF?", "RF" in returns.columns)

In [ ]:
# the three Fama French factors (Mkt-RF already has the risk free rate taken off)
ff3 = factors_log[["Mkt-RF", "SMB", "HML"]]

# excess return of each stock = its return minus the risk free rate, same as the factors
excess = returns.sub(factors_log["RF"], axis=0)

# cumulative plot just to see the factors look sensible (market should trend up)
(ff3.cumsum() / 252).plot(figsize=(10, 3), title="cumulative factor log returns")
plt.show()

The 4-factor model (Fama-French 3 factors + Momentum) is sometimes referred to as 'Carhart' model.

In [ ]:
# Carhart = Fama French 3 + momentum
carhart = factors_log[["Mkt-RF", "SMB", "HML", "Mom"]]
carhart.describe().loc[["mean", "std"]]

To check that we have done everything correctly, let us run a regression of (current) returns based on the factors.
Note that this is not prediction, but instead a model aiming to *explain* current returns by the factor model (Fama-French-3-Factor or Carhart model).

In [ ]:
# train / test split by DATE, same cutoff as the lecture
CUTOFF = "2019-08-31"
split = int((returns.index <= CUTOFF).sum())   # number of training days
print(f"training days: {split} (to {returns.index[split - 1].date()}), "
      f"test days: {len(returns) - split} (from {returns.index[split].date()})")

# one regression per stock, all at once:
# sklearn's LinearRegression accepts a 2D target (days x stocks) and fits each column separately,
# and r2_score then averages the R2 over the stocks
def fit_and_r2(X, Y):
    model = LinearRegression().fit(X, Y)
    return model, r2_score(Y, model.predict(X))

Fit a Fama-French three factor model.

In [ ]:
# EXPLAINING today's excess returns with today's factors (not a forecast), on the training years
model_ff3, r2_ff3 = fit_and_r2(ff3[:split], excess[:split])
print(f"Fama French 3 factor, average R2 across stocks: {r2_ff3:.3f}")

# the average hides a lot of spread, so look at the R2 stock by stock too
r2_each = r2_score(excess[:split], model_ff3.predict(ff3[:split]), multioutput="raw_values")
plt.hist(r2_each, bins=40)
plt.xlabel("R2 per stock")
plt.show()

A 37\% $R^²$ is  reasonable when applying the Fama-French 3-factor model, since it was originally designed to explain cross-sectional average returns (portfolios or sorted groups), not to perfectly predict daily returns of individual stocks.

Fit a carhart four factor model. Note that $R^2$ improves.

In [ ]:
model_c4, r2_c4 = fit_and_r2(carhart[:split], excess[:split])
print(f"Carhart 4 factor, average R2: {r2_c4:.3f}  (3 factor was {r2_ff3:.3f})")

# market beta of each stock is the coefficient on Mkt-RF (first column)
betas = pd.Series(model_c4.coef_[:, 0], index=excess.columns)
print(f"median market beta: {betas.median():.2f}")

## First approaches to returns forecasting

Let us now turn to the problem of forecasting return: try to predict the returns on the next day based on the current Carhart factors and/or returns.

First, let us use the current Fama-French factors to predict next-day returns:

In [ ]:
def forecast_r2(X, Y):
    """X on day t predicts Y on day t+1. Fit on the training days, score on the test days."""
    X, Y = np.asarray(X)[:-1], np.asarray(Y)[1:]   # shift so the features are always one day older
    n = split - 1                                   # samples whose target day is still in training
    model = LinearRegression().fit(X[:n], Y[:n])
    print(f"train R2 {r2_score(Y[:n], model.predict(X[:n])):.4f}, "
          f"test R2 {r2_score(Y[n:], model.predict(X[n:])):.4f}")
    return model

# today's Fama French factors -> tomorrow's returns
forecast_r2(ff3, returns);

This does not look convincing.

Can we improve this by

- including also the momentum factor?
- using a feedforward neural network?
- using a recurrent neural network as in the first part of the lecture?

-------------


Let us try out different alternatives.
##### 1. Including the momentum factor

In [ ]:
# same again, adding momentum
forecast_r2(carhart, returns);

##### 2. Forecasting based on current returns (of all stocks)

In [ ]:
# today's returns of ALL 444 stocks -> tomorrow's returns of all stocks
# 444 inputs for every output, so this has lots of room to overfit
forecast_r2(returns, returns);

##### 3. Forecasting based on current returns (of all stocks) and current factors

In [ ]:
# today's returns of all stocks + today's factors
# (the lecture also passes in the risk free rate, so 444 + 5 = 449 inputs)
factor_cols = ["Mkt-RF", "SMB", "HML", "RF", "Mom"]
forecast_r2(np.hstack([returns, factors_log[factor_cols]]), returns);

##### 4. Using a neural network instead of a linear model

In [ ]:
# a feedforward network in pytorch, same shape as the lecture's Keras one:
# 449 inputs -> 100 -> 200 -> 444 outputs, with 20% dropout after each hidden layer

def train(model, X, y, epochs, lr=1e-3, batch_size=64, seed=1):
    """Generic training loop (used for the feedforward net and every RNN below)."""
    torch.manual_seed(seed)          # fixes the batch shuffling
    loader = DataLoader(TensorDataset(torch.tensor(X), torch.tensor(y)),
                        batch_size=batch_size, shuffle=True)
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    losses = []
    model.train()
    for epoch in range(epochs):
        total = 0.0
        for xb, yb in loader:
            opt.zero_grad()              # clear old gradients
            loss = loss_fn(model(xb), yb)
            loss.backward()              # backprop
            opt.step()                   # update weights
            total += loss.item() * len(xb)
        losses.append(total / len(loader.dataset))
    return losses

def predict(model, X):
    model.eval()                         # switches dropout OFF for predictions
    with torch.no_grad():
        out = model(torch.tensor(X)).numpy()
    return out

In [ ]:
X_ff = np.hstack([returns, factors_log[factor_cols]]).astype(np.float32)[:-1]
Y_ff = returns.values.astype(np.float32)[1:]
n = split - 1

torch.manual_seed(1)
mlp = nn.Sequential(nn.Linear(X_ff.shape[1], 100), nn.ReLU(), nn.Dropout(0.2),
                    nn.Linear(100, 200), nn.ReLU(), nn.Dropout(0.2),
                    nn.Linear(200, Y_ff.shape[1]))
train(mlp, X_ff[:n], Y_ff[:n], epochs=15, batch_size=128)

print(f"feedforward train R2 {r2_score(Y_ff[:n], predict(mlp, X_ff[:n])):.4f}, "
      f"test R2 {r2_score(Y_ff[n:], predict(mlp, X_ff[n:])):.4f}")

None of the above approaches seems to give good results - so let us try to use past (not just current) data for forecasting returns.

#### 5. Building and training a recurrent neural network for prediction

#### Creating lagged data

In [ ]:
# the features for every day: the 444 stock returns + the 5 factor columns = 449
features = np.hstack([returns.values, factors_log[factor_cols].values]).astype(np.float32)
targets = returns.values.astype(np.float32)

WINDOW = 50   # each sample looks back 50 trading days (about 2.5 months)
print(features.shape)

In [ ]:
# lagged samples: 50 days of features -> next day's returns
X, y, t_idx = make_windows(features, targets, window=WINDOW, horizon=1)
print(X.shape, y.shape)  # (samples, 50 days, 449 features), (samples, 444 stocks)

In [ ]:
# split by the date of the LAST input day: if the target day is still before the cutoff it is training
train_mask = t_idx < split - 1
X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[~train_mask], y[~train_mask]
print(X_train.shape, X_test.shape)

#### Building and training a recurrent neural network

We now start to build the recurrent neural network in pytorch. The documentation for simple recurrent neural networks is here:

https://pytorch.org/docs/stable/generated/torch.nn.RNN.html


#### Simple RNN intialized from the $0$ vector.

In [ ]:
class RNNModel(nn.Module):
    """Same as the lecture: a 2 layer RNN (40 hidden units, hidden state starts at zero),
    then 40 -> 30 -> 444 fully connected layers on the last day of the window."""

    def __init__(self, n_features, n_outputs, hidden=40, layers=2):
        super().__init__()
        # batch_first=True means inputs are (batch, time, features)
        self.rnn = nn.RNN(n_features, hidden, num_layers=layers, batch_first=True, dropout=0.1)
        self.fc1 = nn.Linear(hidden, 30)
        self.fc2 = nn.Linear(30, n_outputs)

    def forward(self, x):
        out, _ = self.rnn(x)     # no h0 passed, so pytorch starts from the zero vector
        last = out[:, -1, :]     # output after the last day in the window
        return self.fc2(torch.relu(self.fc1(last)))

Here we use one RNN followed by two fully connected layers.

In [ ]:
torch.manual_seed(1)
rnn = RNNModel(n_features=X.shape[2], n_outputs=y.shape[1])
print(rnn)
print("parameters:", sum(p.numel() for p in rnn.parameters()))

Here we used DataLoader object. It automatically handles batching your data, so you don’t have to manually slice tensors. It also supports shuffling to avoid overfitting to data order.

In [ ]:
# DataLoader hands the model shuffled mini batches of 64 windows
# (the train() function above builds one internally, this is just to look at a batch)
loader = DataLoader(TensorDataset(torch.tensor(X_train), torch.tensor(y_train)), batch_size=64, shuffle=True)
xb, yb = next(iter(loader))
print(xb.shape, yb.shape)

In [ ]:
losses = train(rnn, X_train, y_train, epochs=10)

In [ ]:
plt.plot(losses)
plt.xlabel("epoch")
plt.ylabel("training MSE")
plt.show()

In [ ]:
def information_coefficient(pred, true):
    """Average over days of the rank correlation between predicted and realised returns across stocks.
    This only cares about the ORDER of the stocks each day, which is exactly what a long/short
    strategy uses."""
    p = pd.DataFrame(pred).rank(axis=1)
    t = pd.DataFrame(true).rank(axis=1)
    return p.corrwith(t, axis=1).mean()

pred_test = predict(rnn, X_test)
print(f"RNN train R2 {r2_score(y_train, predict(rnn, X_train)):.4f}, test R2 {r2_score(y_test, pred_test):.4f}")
print(f"RNN test information coefficient: {information_coefficient(pred_test, y_test):.4f}")

**Question: does this mean that the predictions are useless?**

_____

**My answer:** not necessarily. $R^2$ punishes the squared error in the *size* of each daily return, and daily returns are mostly noise, so almost any model scores close to zero or below. A strategy that ranks stocks only needs the *ordering* across stocks to be a bit better than random. That is why I also print the information coefficient (the rank correlation between predicted and realised returns): a small positive IC can still be worth trading. The real test is the backtest below.

# Assessing prediction quality based on a simple trading strategy

Is the $R^2$ the right measure of prediction performance in this context? Let us do another comparison of different return prediction methods by assessing how well they perform when used in a simple trading strategy.

Below we implement such a strategy. It works as follows: we start with fixed initial capital (the size of which is in fact irrelevant). We rebalance at a given frequency (say every 20 trading dates). At each of the rebalancing dates, first we clear our position. This leaves us with a certain amount of money (total_cap). Then we use a prediction model that, based on past and current data (up to a pre-specified lookback window), predicts the average returns for each stock over the next 15 days. Then we rank the stocks according to this indicator and buy those that our model predicts to perform best and short-sell those stocks that the model predicts to perform worst.
The positions we take in these stocks are equally weighted.

In [ ]:
# strategy settings (same as the lecture)
HORIZON = 15      # the model predicts the AVERAGE return over the next 15 days
REBALANCE = 20    # we rebalance every 20 trading days
QUANTILE = 0.05   # long the top 5% of stocks, short the bottom 5% (22 of 444)
N_SHIFTS = 9      # average every Sharpe over 9 different start days, so one lucky
                  # (or unlucky) set of rebalancing dates doesn't drive the result

# long_short_backtest (in src/backtest.py) does the bookkeeping:
# at each rebalance close everything, buy half the capital's worth of the top names and short
# the same amount of the bottom names, equal money in each, hold until the next rebalance

#### Data prepration: predicting average future returns

We first create training data: as features we use time series of a given lenght, as targets we use the mean returns over the next 15 days.

In [ ]:
# new targets: mean return over the next 15 days
X, y, t_idx = make_windows(features, targets, window=WINDOW, horizon=HORIZON)
print(X.shape, y.shape)

In [ ]:
# careful with the split: a sample ending on day t has a target that runs to day t+15,
# so samples ending in the last 15 days before the cutoff would peek into the test period.
# we drop those (an "embargo") so the test period is genuinely unseen.
train_mask = t_idx < split - HORIZON
test_mask = t_idx >= split
X_train, y_train = X[train_mask], y[train_mask]
X_test, y_test = X[test_mask], y[test_mask]
print(X_train.shape, X_test.shape)

#### Trading based on the simple RNN predictor
First we use the RNN predictor with fixed initial state from above. As we are now aiming to predict average future returns, we retrain the RNN.

In [ ]:
torch.manual_seed(1)
rnn = RNNModel(n_features=X.shape[2], n_outputs=y.shape[1])
losses = train(rnn, X_train, y_train, epochs=6)

In [ ]:
plt.plot(losses)
plt.xlabel("epoch")
plt.ylabel("training MSE")
plt.show()

In [ ]:
pred_test = predict(rnn, X_test)
print(f"test R2 {r2_score(y_test, pred_test):.4f}, test IC {information_coefficient(pred_test, y_test):.4f}")

Next we implement the trading strategy as explained above.

In [ ]:
def scores_for(model_predict, idx):
    """Predictions for every stock on each rebalance day, as a DataFrame the backtest can use.
    model_predict takes windows of shape (N, WINDOW, features) and returns (N, stocks).
    Only uses data up to and including the rebalance day itself."""
    windows = np.stack([features[t - WINDOW + 1:t + 1] for t in idx])
    preds = model_predict(windows)
    return pd.DataFrame(preds, index=returns.index[idx], columns=returns.columns)

def backtest(model_predict, first_day, cost_bps=0.0):
    """Runs the strategy N_SHIFTS times, starting on first_day, first_day + 1, ...
    Returns the average Sharpe (measured between rebalances, like the lecture) and the
    account value of the first run (for plots)."""
    sharpes, first_value = [], None
    for s in range(N_SHIFTS):
        idx = np.arange(first_day + s, len(returns), REBALANCE)
        value = long_short_backtest(prices, scores_for(model_predict, idx),
                                    quantile=QUANTILE, cost_bps=cost_bps)
        sharpes.append(sharpe_ratio(value, every=REBALANCE))
        if first_value is None:
            first_value = value
    return np.mean(sharpes), first_value

rnn_predict = lambda w: predict(rnn, w)

In [ ]:
# trade over the WHOLE 13 years first (most of these dates were used to train the RNN!)
sr_all, value_all = backtest(rnn_predict, first_day=WINDOW)
print(f"Sharpe ratio, whole sample: {sr_all:.2f}")

value_all.plot(figsize=(10, 3), title="RNN long/short, whole sample")
plt.axvline(returns.index[split], color="red", ls="--", label="test period starts")
plt.legend()
plt.show()

Let us now evaluate how well the RNN-based strategy performs. We look at varying starting dates.

In [ ]:
# how does the Sharpe depend on when we start trading?
# (the later the start, the more of the run is out of sample)
rows = []
for year in range(2012, 2023):
    first = int(np.searchsorted(returns.index, pd.Timestamp(f"{year}-01-01")))
    rows.append((year, backtest(rnn_predict, first_day=first)[0]))

pd.DataFrame(rows, columns=["start year", "Sharpe"]).set_index("start year").round(2).T

We missed something: we started trading in-sample, which has been used to train the RNN predictor. Now let us measure only the out of sample performance.

In [ ]:
# only trade on dates the RNN has never seen
sr_oos, value_oos = backtest(rnn_predict, first_day=split)

# the lecture started at y_train.shape[0], which is the number of training WINDOWS (cutoff - 50),
# not the cutoff itself, so its first one or two rebalances were still in the training period.
# this reproduces that start, to see how much it matters
sr_lecture, _ = backtest(rnn_predict, first_day=split - WINDOW + REBALANCE)

n_days = len(value_oos) - 1
se = sharpe_standard_error(sr_oos, n_days)
print(f"in sample (whole period) Sharpe:  {sr_all:.2f}")
print(f"out of sample, lecture start:     {sr_lecture:.2f}")
print(f"out of sample, from the cutoff:   {sr_oos:.2f}  "
      f"(rough standard error {se:.2f} over {n_days / 252:.1f} years)")

value_oos.plot(figsize=(10, 3), title="RNN long/short, out of sample only")
plt.show()

# keep the results to compare all the strategies at the end
results = {"RNN (seed 1)": sr_oos}

#### Trading based on alternative predictors

As a further comparison we will now look at the same type of prediction-model-based strategy, but with other types of predictors.

##### 1. Linear regression with flattened time series

First we use a linear model instead of an RNN. Our training data points consist of time series (lagged). A first naive approach just flattens these arrays.

In [ ]:
# flatten each 50 x 449 window into one long vector of 22,450 numbers
flatten = lambda w: w.reshape(len(w), -1)

lin_flat = LinearRegression().fit(flatten(X_train), y_train)
print("number of inputs:", flatten(X_train).shape[1], "| training samples:", len(X_train))
# far more inputs than samples, so the fit can match the training data perfectly -> overfitting

In [ ]:
pred = lin_flat.predict(flatten(X_test))
print(f"train R2 {lin_flat.score(flatten(X_train), y_train):.4f}")
print(f"test R2 {r2_score(y_test, pred):.4f}, test IC {information_coefficient(pred, y_test):.4f}")

In [ ]:
def oos_sharpe(model_predict, name):
    """Out of sample backtest for any predictor, stores the Sharpe in `results`."""
    results[name] = backtest(model_predict, first_day=split)[0]
    print(f"{name}: out of sample Sharpe {results[name]:.2f}")

oos_sharpe(lambda w: lin_flat.predict(flatten(w)), "linear, flattened window")

##### 2. Linear regression based on past average returns
The linear model did not do  well. Let us see if we can improve this by a smarter choice of features. The features now consist of past average returns for each of the stocks (and Carhart factors).

In [ ]:
# smarter features: the AVERAGE of each feature over the window
# (average past return of every stock + average factors) -> 449 inputs instead of 22,450
window_mean = lambda w: w.mean(axis=1)

In [ ]:
lin_avg = LinearRegression().fit(window_mean(X_train), y_train)
pred = lin_avg.predict(window_mean(X_test))
print(f"test R2 {r2_score(y_test, pred):.4f}, test IC {information_coefficient(pred, y_test):.4f}")

In [ ]:
oos_sharpe(lambda w: lin_avg.predict(window_mean(w)), "linear, window averages")

##### 3. Linear regression based on current returns and factors

Using the average over the recent past seems to work relatively well. As a comparison, let us also try to use just the previous returns and factors (not the average over the recent past).

In [ ]:
# only the most recent day of the window: the latest returns and factors
last_day = lambda w: w[:, -1, :]

In [ ]:
lin_last = LinearRegression().fit(last_day(X_train), y_train)
pred = lin_last.predict(last_day(X_test))
print(f"test R2 {r2_score(y_test, pred):.4f}, test IC {information_coefficient(pred, y_test):.4f}")

In [ ]:
oos_sharpe(lambda w: lin_last.predict(last_day(w)), "linear, current returns + factors")

##### 4. Linear regression based on current factors

Using only the previous factors and returns seemed to works quite a bit less well. Finally, let us try to use as features only the Carhart factors, not the returns.

In [ ]:
# only the 5 factor columns from the last day (they are the last 5 columns of the features)
# like the lecture, this one is fitted to predict NEXT DAY returns from today's factors
factors_only = lambda w: w[:, -1, -5:]
lin_fac = LinearRegression().fit(features[:split - 1, -5:], targets[1:split])

In [ ]:
oos_sharpe(lambda w: lin_fac.predict(factors_only(w)), "linear, current factors only")

Compare the four linear models with the RNN from above: which features carry the useful information?

### Sensitivity to randomness in RNN training

Finally, let us assess how sensitive these results are to the random initialization of RNN training. To do this let us analyse the out-of-sample performance for 20 different RNNs.

In [ ]:
# train 20 RNNs that differ ONLY in their random seed (initial weights + batch order)
# this is the slow cell: 20 trainings + 9 backtests each
seed_sharpes = []
for seed in range(20):
    torch.manual_seed(seed)
    model = RNNModel(n_features=X.shape[2], n_outputs=y.shape[1])
    train(model, X_train, y_train, epochs=6, seed=seed)
    sr, _ = backtest(lambda w: predict(model, w), first_day=split)
    seed_sharpes.append(sr)
    print(f"seed {seed:2d}: out of sample Sharpe {sr:.2f}")

Let us print the list of Sharpe ratios for the different RNNs:

In [ ]:
seed_sharpes = pd.Series(seed_sharpes, name="out of sample Sharpe")
print(seed_sharpes.round(2).tolist())
print(f"median {seed_sharpes.median():.2f}, mean {seed_sharpes.mean():.2f}, "
      f"min {seed_sharpes.min():.2f}, max {seed_sharpes.max():.2f}")

In [ ]:
# how do the 20 RNNs compare with the linear models?
plt.hist(seed_sharpes, bins=10, alpha=0.7, label="20 RNN seeds")
for name, sr in results.items():
    if name.startswith("linear"):
        plt.axvline(sr, color="black", ls="--", lw=1)
plt.xlabel("out of sample Sharpe")
plt.title("dashed lines = linear benchmarks")
plt.legend()
plt.show()

results["RNN, median of 20 seeds"] = seed_sharpes.median()
pd.Series(results).round(2).to_frame("out of sample Sharpe")

We see that there is some variation in Sharpe ratios across the 20 seeds. Compare the median with the linear baselines in the table above.

## My extensions: is the Sharpe ratio real?

Before believing any of these Sharpe ratios I want to check three things:

1. **Error bars.** The test period is only about four years, so a Sharpe ratio estimated from it is noisy (the standard error is printed above).
2. **Luck.** If I rank the stocks completely at random, what Sharpe ratios come out of the same backtest?
3. **Costs.** Every rebalance we turn over the whole book. How much of the edge survives a realistic trading cost?

In [ ]:
# 2. random rankings: same strategy and dates, but the scores are pure noise
rng = np.random.default_rng(0)
idx = np.arange(split, len(returns), REBALANCE)
random_sharpes = []
for i in range(200):
    noise = pd.DataFrame(rng.normal(size=(len(idx), returns.shape[1])),
                         index=returns.index[idx], columns=returns.columns)
    value = long_short_backtest(prices, noise, quantile=QUANTILE)
    random_sharpes.append(sharpe_ratio(value, every=REBALANCE))
random_sharpes = np.array(random_sharpes)

plt.hist(random_sharpes, bins=30, alpha=0.7, label="random rankings")
plt.axvline(seed_sharpes.median(), color="red", label="RNN median")
plt.xlabel("out of sample Sharpe")
plt.legend()
plt.show()

# share of random strategies doing at least as well as the median RNN (a rough p value)
print(f"random strategies beating the median RNN: {(random_sharpes >= seed_sharpes.median()).mean():.1%}")
print(f"median RNN vs random: z = {(seed_sharpes.median() - random_sharpes.mean()) / random_sharpes.std():.2f}")

In [ ]:
# 3. trading costs: rerun the seed 1 RNN with a cost per trade, in basis points
for cost in [0, 2, 5, 10]:
    sr, _ = backtest(rnn_predict, first_day=split, cost_bps=cost)
    print(f"cost {cost:2d} bp: Sharpe {sr:.2f}")